In [ ]:
#Get SPI extremes stats based on thresholds
import rasterio
import numpy as np
import pandas as pd
with rasterio.open(f"spi_month_timescale012_statewide_data_map_2025_12.tif") as src:
    data = src.read(1)
    profile = src.profile
    nodata = src.nodata

valid = np.isfinite(data)

pct = np.sum((data < -1.3) & valid) / np.sum(valid) * 100
print(f"{pct:.2f}%")

In [ ]:
#SPI12 thresholds by county
import numpy as np
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

shapefile = "../../public/shapefiles/islands.shp"   # has 'name' for county
raster_path = "spi_month_timescale012_statewide_data_map_2025_12.tif" 

# one polygon per county
gdf = gpd.read_file(shapefile).dissolve(by="name", as_index=False).reset_index(drop=True)

threshold = -0.8

with rasterio.open(raster_path) as src:
    arr = src.read(1, masked=True)
    nodata = src.nodata
    affine = src.transform

# boolean raster: 1 where condition is met, 0 elsewhere (masked nodata stays masked)
cond = (arr < threshold).astype("uint8")
cond = np.ma.array(cond, mask=arr.mask)

# zonal sums of condition pixels and counts of valid pixels
sum_stats = zonal_stats(
    gdf, cond.filled(0), affine=affine,
    stats=["sum"], nodata=0, all_touched=False
)
cnt_stats = zonal_stats(
    gdf, np.where(arr.mask, 0, 1).astype("uint8"), affine=affine,
    stats=["sum"], nodata=0, all_touched=False
)

gdf["n_lt_thr"] = [s["sum"] or 0 for s in sum_stats]
gdf["n_valid"]  = [c["sum"] or 0 for c in cnt_stats]

gdf[f"pct_{threshold}"] = np.where(
    gdf["n_valid"] > 0,
    100.0 * gdf["n_lt_thr"] / gdf["n_valid"],
    np.nan
)
gdf[f"pct_{threshold}"] = gdf[f"pct_{threshold}"].round(0).astype("Int64").astype(str) + "%"

gdf[["name", f"pct_{threshold}"]].to_csv(f"island_spi12_below_{threshold}_2025.csv", index=False)
print(gdf[["name", f"pct_{threshold}"]])